# Customer Churn Prediction
**Goal:** Predict which telecom customers will churn, and explain *why* using SHAP.

**Dataset:** Telco Customer Churn (Kaggle) — 7,043 customers, 21 features

**Stack:** pandas · scikit-learn Pipeline · SHAP · matplotlib · joblib

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('All imports OK')

## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print(f'\nChurn rate: {df["Churn"].value_counts(normalize=True).round(3)}')

## 3. Data Cleaning
> **Key gotcha:** `TotalCharges` is stored as a string. Fix it before anything else.

In [ ]:
# Fix TotalCharges — convert to numeric, blank strings become NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 11 rows have blank TotalCharges — fill with median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Convert target to binary
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Drop customerID — not a feature
df.drop(columns=['customerID'], inplace=True)

print('Cleaning done. Churn distribution:')
print(df['Churn'].value_counts())

## 4. Exploratory Data Analysis (EDA)
Focus on the three strongest churn signals: **contract type, tenure, monthly charges**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Churn by Contract Type
contract_churn = df.groupby('Contract')['Churn'].mean().reset_index()
axes[0].bar(contract_churn['Contract'], contract_churn['Churn'] * 100, color=['#1D9E75','#5B5EA6','#E87B51'])
axes[0].set_title('Churn Rate by Contract Type', fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_xlabel('')

# 2. Tenure distribution by Churn
df[df['Churn']==0]['tenure'].hist(ax=axes[1], alpha=0.6, label='Stayed', bins=30, color='#1D9E75')
df[df['Churn']==1]['tenure'].hist(ax=axes[1], alpha=0.6, label='Churned', bins=30, color='#E87B51')
axes[1].set_title('Tenure Distribution by Churn', fontweight='bold')
axes[1].set_xlabel('Tenure (months)')
axes[1].legend()

# 3. Monthly Charges by Churn
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[2], 
           boxprops=dict(color='#1D9E75'), medianprops=dict(color='#E87B51', linewidth=2))
axes[2].set_title('Monthly Charges by Churn', fontweight='bold')
axes[2].set_xlabel('Churn (0=Stayed, 1=Churned)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../eda_key_signals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to eda_key_signals.png — add this to your README!')

In [ ]:
# Key finding — print it clearly
mtm_churn = df[df['Contract']=='Month-to-month']['Churn'].mean()
two_yr_churn = df[df['Contract']=='Two year']['Churn'].mean()
print(f'Month-to-month churn rate: {mtm_churn:.1%}')
print(f'Two-year contract churn rate: {two_yr_churn:.1%}')
print(f'Month-to-month customers churn {mtm_churn/two_yr_churn:.1f}x more than two-year customers')

## 5. Feature Engineering & Train/Test Split

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

# Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f'Numeric features ({len(numeric_cols)}): {numeric_cols}')
print(f'Categorical features ({len(categorical_cols)}): {categorical_cols}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

## 6. Build the scikit-learn Pipeline
> This is the **modern way** — preprocessing + model in one object. No data leakage. Reproducible.

In [ ]:
# Preprocessing
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

# --- Baseline: Logistic Regression ---
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

lr_auc = cross_val_score(lr_pipeline, X_train, y_train, cv=5, scoring='roc_auc').mean()
print(f'Logistic Regression CV ROC-AUC: {lr_auc:.4f}')

In [ ]:
# --- Random Forest ---
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1))
])

rf_auc = cross_val_score(rf_pipeline, X_train, y_train, cv=5, scoring='roc_auc').mean()
print(f'Random Forest CV ROC-AUC:       {rf_auc:.4f}')

In [ ]:
# --- Gradient Boosting (usually wins) ---
gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42))
])

gb_auc = cross_val_score(gb_pipeline, X_train, y_train, cv=5, scoring='roc_auc').mean()
print(f'Gradient Boosting CV ROC-AUC:   {gb_auc:.4f}')

print(f'\n✅ Best model: {"Gradient Boosting" if gb_auc >= rf_auc else "Random Forest"}')

## 7. Evaluate Best Model on Test Set

In [ ]:
# Train final model on full training set
best_pipeline = gb_pipeline  # swap to rf_pipeline if RF wins above
best_pipeline.fit(X_train, y_train)

y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

print(f'Test ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Stayed','Churned']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Stayed','Churned'],
    cmap='Greens', ax=axes[0]
)
axes[0].set_title('Confusion Matrix', fontweight='bold')

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1], color='#1D9E75')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].plot([0,1],[0,1],'--', color='gray', alpha=0.5)

plt.tight_layout()
plt.savefig('../evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. SHAP Explainability
> This is what separates a good project from a great one. Shows *why* the model predicts what it predicts.

In [ ]:
import shap

# Get preprocessed test data
X_test_transformed = best_pipeline.named_steps['preprocessor'].transform(X_test)

# Get feature names after one-hot encoding
ohe_features = best_pipeline.named_steps['preprocessor']\
    .named_transformers_['cat'].get_feature_names_out(categorical_cols).tolist()
all_features = numeric_cols + ohe_features

# SHAP explainer
explainer = shap.TreeExplainer(best_pipeline.named_steps['model'])
shap_values = explainer.shap_values(X_test_transformed)

print('SHAP values computed successfully')
print(f'Shape: {shap_values.shape}')

In [ ]:
# Summary plot — top features driving churn
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_test_transformed,
    feature_names=all_features,
    max_display=15, show=False
)
plt.title('SHAP Feature Importance — What Drives Churn?', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP summary saved — this goes in your README!')

In [ ]:
# Waterfall plot for one high-risk customer
high_risk_idx = y_prob.argmax()  # the customer most likely to churn

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[high_risk_idx],
        base_values=explainer.expected_value,
        data=X_test_transformed[high_risk_idx],
        feature_names=all_features
    )
)
plt.title('Why is this customer likely to churn?', fontweight='bold')
plt.tight_layout()
plt.savefig('../shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Business Insights
> Translate model findings into plain English. This is what non-technical hiring managers read.

In [ ]:
mtm_churn = df[df['Contract']=='Month-to-month']['Churn'].mean()
two_yr_churn = df[df['Contract']=='Two year']['Churn'].mean()
new_cust_churn = df[df['tenure'] <= 12]['Churn'].mean()

print('=== KEY BUSINESS FINDINGS ===')
print(f'1. Contract type is the #1 churn driver.')
print(f'   Month-to-month: {mtm_churn:.1%} churn vs Two-year: {two_yr_churn:.1%} churn')
print(f'   → Prioritize annual plan incentives for new sign-ups\n')
print(f'2. New customers are highest risk.')
print(f'   Customers in first 12 months: {new_cust_churn:.1%} churn rate')
print(f'   → Invest in onboarding and early engagement programs\n')
print(f'3. High monthly charges correlate with churn.')
print(f'   → Consider loyalty discounts for high-spend customers at risk')

## 10. Save Model

In [ ]:
joblib.dump(best_pipeline, '../models/churn_model.pkl')
print('Model saved to models/churn_model.pkl')

# Verify it loads correctly
loaded = joblib.load('../models/churn_model.pkl')
test_score = roc_auc_score(y_test, loaded.predict_proba(X_test)[:,1])
print(f'Verified — loaded model ROC-AUC: {test_score:.4f}')
print('\n✅ Project complete. Push to GitHub!')